In [1]:
import json
from itertools import batched

import numpy as np
import torch
import torch.nn.functional as F
from datasets import Array3D, Dataset, Features, Sequence, Value
from transformers import AutoModel, AutoTokenizer
from utils.data import Cst

MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B"

PADDING = 400
BATCH_SIZE = 100

device = torch.device("mps")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

N_LAYERS = model.config.num_hidden_layers
D_MODEL = model.config.hidden_size

with open("../trees.json", "r") as file:
    data = [Cst.from_dict(sample) for sample in json.load(file)]

/Users/hw452/Documents/programming/structural-probe-ast-rewrite/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1120 12:12:02.668000 31186 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9726.58it/s]


In [ ]:
def pad_distance(
    distance: torch.tensor, pad_to: int = PADDING
) -> tuple[torch.tensor, torch.tensor]:
    N, _ = distance.shape

    mask = torch.full((pad_to,), 0, dtype=torch.bool)
    output = torch.full((pad_to, pad_to), 0, dtype=distance.dtype)

    output[:N, :N] = distance
    mask[:N] = 1

    return output, mask


distance = torch.tensor(data[0].distance)
D, M = pad_distance(distance)

assert (distance == D[M][:, M]).all()

In [ ]:
def pad_depth(
    depth: torch.tensor, pad_to: int = PADDING
) -> tuple[torch.tensor, torch.tensor]:
    (N,) = depth.shape

    mask = torch.full((pad_to,), 0, dtype=torch.bool)
    output = torch.full((pad_to,), 0, dtype=depth.dtype)

    output[:N] = depth
    mask[:N] = 1

    return output, mask.to(torch.bool)


depth = torch.tensor(data[0].depth[data[0].leaf_indicies])
D, M = pad_depth(depth)

assert (depth == D[M]).all()

In [ ]:
from dataclasses import dataclass


@dataclass
class Item:
    # (max_seq_length, max_seq_length)
    distance: torch.Tensor

    # (max_seq_length,)
    depth: torch.Tensor

    # (max_seq_length,)
    mask: torch.Tensor

    # (seq_length)
    tokens: list[str]

    # (num_layers, max_seq_length, d_model)
    hidden_representation: torch.Tensor

    language: str

In [ ]:
def process_batch(batch):
    batch_size = len(batch)

    distances = []
    depths = []
    masks = []
    fragments = []
    languages = []
    for item in batch:
        if item.num_leaves > PADDING:
            batch_size -= 1
            continue

        distance, mask = pad_distance(torch.tensor(item.distance))
        depth, _ = pad_depth(torch.tensor(item.depth[item.leaf_indicies]))

        distances.append(distance)
        depths.append(depth)
        masks.append(mask)
        fragments.append([leaf.text for leaf in item.leaves])
        languages.append(item.language)

    if not distances:
        return []

    distances = torch.stack(distances)
    depths = torch.stack(depths)
    masks = torch.stack(masks)

    enc = tokenizer(
        fragments,
        is_split_into_words=True,
        max_length=PADDING,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        output = model(**enc, output_hidden_states=True)

    H = torch.stack(output.hidden_states)  # (L=25, B, T, D)
    word_ids = torch.tensor(
        [
            [wid if wid is not None else -1 for wid in enc.word_ids(i)]
            for i in range(batch_size)
        ]
    ).to(device)  # (B, T), -1 = special/pad token

    # A[b, t, w] = 1/count(w) if token t belongs to word w
    A = F.one_hot(word_ids.clamp(min=0), num_classes=PADDING).to(H.dtype)  # (B, T, W)
    A = A * (word_ids >= 0).unsqueeze(-1)  # drop special/pad tokens
    A = A / A.sum(dim=1, keepdim=True).clamp(min=1)  # mean over subwords

    pooled = torch.einsum("btw,lbtd->lbwd", A, H)  # (L, B, W, D)
    pooled = pooled.cpu()

    items = [
        Item(
            distance=distances[b],
            depth=depths[b],
            mask=masks[b],
            tokens=fragments[b],
            hidden_representation=pooled[:, b],
            language=languages[b],
        )
        for b in range(batch_size)
        if not max(w for w in enc.word_ids(b) if w is not None) + 1 < len(fragments[b])
    ]
    return items


def dataset_generator(data):
    for batch in batched(data, BATCH_SIZE):
        processed = process_batch(batch)

        for item in processed:
            n = len(item.tokens)
            yield {
                "tokens": item.tokens,
                "distance": item.distance[:n, :n].cpu().numpy().astype(np.int16),
                "depth": item.depth[:n].cpu().numpy().astype(np.int16),
                "hidden_representation": item.hidden_representation[:, :n]
                .permute(1, 0, 2)
                .to(torch.float16)
                .cpu()
                .numpy(),
                "language": item.language,
            }


features = Features(
    {
        "tokens": Sequence(Value("string")),
        "distance": Sequence(Sequence(Value("int16"))),
        "depth": Sequence(Value("int16")),
        "hidden_representation": Array3D(
            shape=(None, N_LAYERS + 1, D_MODEL), dtype="float16"
        ),
        "language": Value("string"),
    }
)
ds = Dataset.from_generator(
    dataset_generator,
    gen_kwargs={"data": data},
    features=features,
    fingerprint=f"{MODEL_NAME.replace('/', '_')}-pad{PADDING}-v2",
)

In [ ]:
ds = ds.train_test_split()

In [ ]:
ds.save_to_disk(f"../dataset/{MODEL_NAME}")